# YOLOv8 방해차량(vehA #46) 파인튜닝 로컬 학습 (Ubuntu + NVIDIA RTX 3070 Ti / CUDA)

`track_drive`의 `yolo_vehicle.py`가 COCO 사전학습 `yolov8n.pt`의 범용 `car` 클래스를
그대로 써서 신뢰도가 낮게 나오는 문제(실측 0.15~0.78, 평균 0.3대)를 고치기 위해,
대회에서 실제로 회피해야 하는 **그 차량 한 대**(#46, TRAXXAS 검정/연두)만 전용으로
검출하는 모델을 만든다.

[`TwinLiteNet-KMU-finetune`](https://github.com/mastic-choi/TwinLiteNet-KMU-finetune)의
`finetune_ensemble_v2_local_rtx.ipynb` 구조(경로 설정 → CUDA 확인 → 데이터 준비 →
하이퍼파라미터 → 학습 → 결과 export)를 그대로 따른다. 다만 그쪽은 **자동 라벨(da/ll
세그멘테이션) 품질을 높이려고** 시드만 다른 모델 5개를 앙상블(soft-vote)했지만,
여기서는 앙상블이 필요 없다 — 시드 라벨이 처음부터 사람이 CVAT에서 박스로 직접
확정한 값이라 "여러 모델의 합의로 라벨 노이즈를 평균화"할 대상 자체가 없다. 단일
모델을 bootstrap 반복(추론→저신뢰만 사람이 보정→재학습)으로 개선하는 쪽이 맞다
(저장소 루트 `README.md` "방법론" 참고).


## 0. 환경 설정 (로컬 경로 + CUDA 확인)

In [ ]:
import os

REPO_DIR = os.path.expanduser('~/yolo-V8-KMU-xycar')
BASE_DIR = os.path.expanduser('~/umk_yolo_vehicle')   # work/runs 산출물은 저장소 밖(용량/gitignore 신경 안 쓰게)

WORK_DIR = os.path.join(BASE_DIR, 'work')
DATA_DIR = os.path.join(REPO_DIR, 'target_vehicle', 'data', 'seed_labeled')   # 시드 라벨링 세트(2127장, images/+labels/ 1:1 대응)
RUNS_DIR = os.path.join(BASE_DIR, 'runs')

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print('DATA_DIR:', DATA_DIR)
assert os.path.isdir(os.path.join(DATA_DIR, 'images')) and os.path.isdir(os.path.join(DATA_DIR, 'labels')), \
    f'{DATA_DIR} 밑에 images/, labels/ 가 없음 - seed_labeled 데이터셋을 먼저 풀어둘 것'


In [ ]:
import subprocess, sys

try:
    import torch
except ImportError:
    # RTX 3070 Ti = Ampere 아키텍처 -> CUDA 12.1 빌드로 설치(2026-08 기준 안정 채널)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                     'torch', 'torchvision',
                     '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    import torch

print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('VRAM(GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('[경고] CUDA 인식 안 됨 - `nvidia-smi`로 드라이버부터 확인할 것')


In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'  # wandb 미로그인 상태면 학습이 no-tty 에러로 죽음 - CVAT/사내망 환경이라 wandb 안 씀

!pip install -q ultralytics
import ultralytics
ultralytics.checks()


## 1. 데이터 준비

CVAT(`app.cvat.ai`, 기존 `UMK` 조직)에서 시드 라벨링을 마친 태스크를 **YOLO 1.1**
포맷으로 export → `DATA_DIR/images/`, `DATA_DIR/labels/` 밑에 이미지·txt 라벨이
같은 파일명(확장자만 다름)으로 1:1 대응되게 풀어둔다. 클래스는 1개(`target_vehicle`).

라벨이 하나도 없는 프레임(음성 샘플, background)도 섞고 싶으면 `labels/`에
대응하는 빈 txt만 두면 된다(YOLO 관례) — 지금 저장소 `data/candidates/clear_rear`
같은 확실한 프레임 위주로 우선 채우고, 저신뢰/미검출 프레임(음성 예시 포함)은
`scripts/scan_dedicated_capture.py` 결과에서 추려 CVAT에 추가로 올릴 것.


In [ ]:
import glob, random

random.seed(42)
all_images = sorted(
    glob.glob(os.path.join(DATA_DIR, 'images', '*.jpg')) +
    glob.glob(os.path.join(DATA_DIR, 'images', '*.png'))
)
assert all_images, f'{DATA_DIR}/images 밑에 이미지가 없음 - CVAT export를 먼저 풀어둘 것'

random.shuffle(all_images)
n_val = max(1, int(len(all_images) * 0.15))
val_images = all_images[:n_val]
train_images = all_images[n_val:]

with open(os.path.join(WORK_DIR, 'train.txt'), 'w') as f:
    f.write('\n'.join(train_images))
with open(os.path.join(WORK_DIR, 'val.txt'), 'w') as f:
    f.write('\n'.join(val_images))

print(f'train {len(train_images)}장 / val {len(val_images)}장')


## 2. data.yaml 작성

In [ ]:
import yaml

data_yaml = {
    'train': os.path.join(WORK_DIR, 'train.txt'),
    'val': os.path.join(WORK_DIR, 'val.txt'),
    'nc': 1,
    'names': ['target_vehicle'],  # vehA(#46) 한 대만 검출. track_drive 쪽 config.py
                                   # YOLO_VEHICLE_CLASS_ID와 이름/순서를 맞출 것(아래 6번 참고)
}
DATA_YAML_PATH = os.path.join(WORK_DIR, 'target_vehicle.yaml')
with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True)
print('저장:', DATA_YAML_PATH)


## 3. 하이퍼파라미터 + 학습 실행

시드 데이터가 아직 100~200장 규모로 적으니 augmentation을 과하게 주지 않는다 —
`mosaic`/`mixup`을 세게 걸면 라벨 노이즈만 키운다(TwinLiteNet-KMU가 작은
코퍼스(156장)에서 커브 바깥까지 과다검출하던 것과 같은 이유 — 그쪽
README "라벨링 품질 개선" 섹션 참고). 코퍼스가 bootstrap 반복으로 커지면
그때 다시 올려도 된다.


In [ ]:
from ultralytics import YOLO

BASE_WEIGHTS = 'yolov8n.pt'  # COCO 사전학습 - track_drive가 이미 쓰는 것과 같은 계열(yolov8n)
EPOCHS = 150
IMG_SIZE = 640
BATCH = 32        # RTX 3070 Ti 8GB 기준. OOM 나면 16으로 낮출 것
SEED = 42

model = YOLO(BASE_WEIGHTS)
model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,          # RTX 3070 Ti 단일 GPU -> CUDA:0
    seed=SEED,
    patience=30,       # 30epoch 연속 개선 없으면 조기종료
    project=RUNS_DIR,
    name='target_vehicle_v1',
    mosaic=0.3,
    mixup=0.0,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,        # 차량 외형상 좌우 대칭 자연스러움 - 문제되면 0으로
    flipud=0.0,
)
print('학습 완료. best.pt:', model.trainer.best)


## 4. 검증

In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)


## 5. ONNX export (track_drive 배포 호환)

`track_drive/perception/yolo_vehicle.py`(`YoloVehicleEngine.infer()`)는
`nms=True`로 export된 ONNX를 전제로, `output0`에서 `[x1,y1,x2,y2,conf,cls]`
형식을 그대로 읽는다(`yolo_ros/yolov8n_car.onnx`와 동일 규약). 지금은 클래스가
1개뿐이라 `cls`는 항상 `0`이 나온다.

실차 배포 타겟은 **reComputer Super J4012(Jetson Orin NX 16GB, JetPack 6,
CUDA 12.6)**. 여기서 만드는 ONNX는 하드웨어 종속이 없어 그대로 옮기면 되지만,
**TensorRT `.engine`으로 바꾸고 싶다면 이 PC(x86 RTX 3070 Ti)에서 만들면 안 된다** -
TensorRT engine은 빌드한 GPU 아키텍처/TensorRT 버전에 고정되는 포맷이라, x86에서
만든 engine을 Jetson(ARM, 다른 GPU 아키텍처)에서 못 읽는다. TensorRT가 필요하면
이 노트북은 ONNX까지만 만들고, `.engine` 변환은 Jetson 보드에 ONNX를 올린 뒤
JetPack 6에 포함된 TensorRT로 그 자리에서(`trtexec` 또는
`YOLO('best.onnx').export(format='engine', ...)`) 해야 한다.


In [ ]:
best_pt = model.trainer.best  # 예: {RUNS_DIR}/target_vehicle_v1/weights/best.pt
best_model = YOLO(best_pt)

onnx_path = best_model.export(
    format='onnx',
    nms=True,     # yolo_ros/yolov8n_car.onnx와 동일 규약 - yolo_vehicle.py가 이 형식을 그대로 기대함
    imgsz=IMG_SIZE,
    opset=12,
    simplify=True,
)
print('ONNX export 완료:', onnx_path)


## 6. 배포 체크리스트 (실차 반영 전 필수)

1. `yolo_ros/`에 이 ONNX를 새 파일명(예: `target_vehicle_best.onnx`)으로 추가하고,
   `config.py`의 `YOLO_VEHICLE_MODEL_PATH`를 이 경로로 지정
   (기존 `yolov8n_car.onnx`를 바로 덮어쓰지 말 것 - 비교/롤백용으로 남겨둘 것)
2. `config.py YOLO_VEHICLE_CLASS_ID`를 `2`(COCO `car`)에서 **`0`**으로 변경 -
   우리 모델은 클래스가 `target_vehicle` 하나뿐이라 id가 0부터 시작
3. `YOLO_VEHICLE_CONF_THRESHOLD` 재조정 - 전용 모델이라 신뢰도 분포 자체가
   기존 COCO 모델과 달라질 가능성이 높음, 새 모델로 정적 이미지 재추론해서
   실측 분포 확인 후 조정할 것
4. (TensorRT를 쓸 경우) ONNX를 reComputer Super J4012(Jetson Orin NX, JetPack 6)로
   옮긴 뒤 그 보드 위에서 `.engine`으로 변환할 것 - 5번 셀 마크다운 참고, x86에서
   만든 engine은 Jetson에서 못 씀. 변환 후 `yolo_vehicle.py`의 추론 코드도
   onnxruntime이 아닌 TensorRT 런타임 호출로 바꿔야 함(런타임 API가 다름)
5. `DEBUG_VIZ_YOLO_VEHICLE=True` 상태로 저속 실차 테스트 - 바운딩박스가
   실제로 vehA에 잘 붙는지, 다른 물체(다른 색 RC카, 사람 등) 오탐은 없는지 확인
   (CLAUDE.md "저장소 = 실차 배포 소스" 관례 - 새 모델 첫 투입은 반드시 실측 검증)
6. bootstrap 다음 라운드 필요하면(README 방법론 3~4단계), 이 노트북의
   `BASE_WEIGHTS`를 이번 `best.pt`로 바꿔서 재학습
